# 04 - O mesmo pipeline em PySpark (tratamento + modelo com Spark MLlib)

Os notebooks 01 e 02 usam Pandas e scikit-learn, que processam tudo na memória de uma máquina.
Em bases grandes (milhões de clientes), o padrão é o **Apache Spark**, que distribui o processamento.
Aqui refaço o tratamento da etapa 01 e treino uma Regressão Logística com o **Spark MLlib**, e comparo com o resultado do scikit-learn.

Roda localmente (`local[*]`) com PySpark 3.5 e Java 8+.

In [1]:
import os, sys
os.environ.setdefault('PYSPARK_PYTHON', sys.executable)
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master('local[*]').appName('churn')
         .config('spark.ui.enabled', 'false').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

df = spark.read.csv('../data/telco_churn.csv', header=True, inferSchema=True)
print(df.count(), 'linhas')
df.printSchema()

7043 linhas
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



## 1. Tratamento

Mesmo problema da etapa 01: `TotalCharges` foi lido como texto (`string`) por causa dos valores em branco.
Converto para número; o que não for número vira `null`, e esses clientes (sem nenhuma fatura) são removidos, a mesma decisão da etapa 01.

In [2]:
df = df.withColumn('TotalCharges', F.trim('TotalCharges').cast('double'))
print('Nulos em TotalCharges:', df.filter(F.col('TotalCharges').isNull()).count())
df.filter(F.col('TotalCharges').isNull()).select('customerID', 'tenure', 'TotalCharges').show()

df = (df.dropna(subset=['TotalCharges'])
        .withColumn('label', (F.col('Churn') == 'Yes').cast('int')))
print('Linhas após tratamento:', df.count())

Nulos em TotalCharges: 11


+----------+------+------------+
|customerID|tenure|TotalCharges|
+----------+------+------------+
|4472-LVYGI|     0|        NULL|
|3115-CZMZD|     0|        NULL|
|5709-LVOEQ|     0|        NULL|
|4367-NUYAO|     0|        NULL|
|1371-DWPAZ|     0|        NULL|
|7644-OMVMY|     0|        NULL|
|3213-VVOLG|     0|        NULL|
|2520-SGTTA|     0|        NULL|
|2923-ARZLG|     0|        NULL|
|4075-WKNIU|     0|        NULL|
|2775-SEFEE|     0|        NULL|
+----------+------+------------+



Linhas após tratamento: 7032


## 2. Análise com agregações do Spark (confere com a etapa 01)

In [3]:
df.groupBy('label').count().withColumn('pct', F.round(F.col('count') / df.count() * 100, 1)).show()

for c in ['Contract', 'InternetService', 'PaymentMethod']:
    (df.groupBy(c).agg(F.round(F.avg('label') * 100, 1).alias('churn_%'), F.count('*').alias('clientes'))
       .orderBy(F.desc('churn_%')).show(truncate=False))

+-----+-----+----+
|label|count| pct|
+-----+-----+----+
|    1| 1869|26.6|
|    0| 5163|73.4|
+-----+-----+----+



+--------------+-------+--------+
|Contract      |churn_%|clientes|
+--------------+-------+--------+
|Month-to-month|42.7   |3875    |
|One year      |11.3   |1472    |
|Two year      |2.8    |1685    |
+--------------+-------+--------+



+---------------+-------+--------+
|InternetService|churn_%|clientes|
+---------------+-------+--------+
|Fiber optic    |41.9   |3096    |
|DSL            |19.0   |2416    |
|No             |7.4    |1520    |
+---------------+-------+--------+



+-------------------------+-------+--------+
|PaymentMethod            |churn_%|clientes|
+-------------------------+-------+--------+
|Electronic check         |45.3   |2365    |
|Mailed check             |19.2   |1604    |
|Bank transfer (automatic)|16.7   |1542    |
|Credit card (automatic)  |15.3   |1521    |
+-------------------------+-------+--------+



In [4]:
faixa = (F.when(F.col('tenure') <= 6, '0-6').when(F.col('tenure') <= 12, '7-12')
          .when(F.col('tenure') <= 24, '13-24').when(F.col('tenure') <= 48, '25-48').otherwise('49-72'))
(df.withColumn('faixa_tenure', faixa).groupBy('faixa_tenure')
   .agg(F.round(F.avg('label') * 100, 1).alias('churn_%'), F.count('*').alias('clientes'), F.min('tenure').alias('_ordem'))
   .orderBy('_ordem').drop('_ordem').show())

+------------+-------+--------+
|faixa_tenure|churn_%|clientes|
+------------+-------+--------+
|         0-6|   53.3|    1470|
|        7-12|   35.9|     705|
|       13-24|   28.7|    1024|
|       25-48|   20.4|    1594|
|       49-72|    9.5|    2239|
+------------+-------+--------+



## 3. Modelo com Spark MLlib

Mesma ideia do scikit-learn, com as peças do MLlib:
- `StringIndexer` + `OneHotEncoder`: categorias → colunas 0/1
- `VectorAssembler` + `StandardScaler`: junta tudo num vetor e padroniza
- **Peso por classe** (`weightCol`): o equivalente ao `class_weight='balanced'`, para compensar os 26,6% de churn
- `LogisticRegression` e avaliação por **recall** e **AUC** num teste 80/20

In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

num = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
cat = [c for c, t in df.dtypes if t == 'string' and c not in ('customerID', 'Churn')]

taxa = df.agg(F.avg('label')).first()[0]
df = df.withColumn('peso', F.when(F.col('label') == 1, 0.5 / taxa).otherwise(0.5 / (1 - taxa)))

treino, teste = df.randomSplit([0.8, 0.2], seed=42)
pipeline = Pipeline(stages=[
    StringIndexer(inputCols=cat, outputCols=[c + '_idx' for c in cat], handleInvalid='keep'),
    OneHotEncoder(inputCols=[c + '_idx' for c in cat], outputCols=[c + '_ohe' for c in cat]),
    VectorAssembler(inputCols=num + [c + '_ohe' for c in cat], outputCol='features_raw'),
    StandardScaler(inputCol='features_raw', outputCol='features'),
    LogisticRegression(featuresCol='features', labelCol='label', weightCol='peso', maxIter=100),
])
modelo = pipeline.fit(treino)
pred = modelo.transform(teste)
print('Treino:', treino.count(), '| Teste:', teste.count())

Treino: 5690 | Teste: 1342


In [6]:
auc = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC').evaluate(pred)
m = pred.groupBy('label', 'prediction').count().toPandas().pivot(index='label', columns='prediction', values='count').fillna(0)
vp, fn, fp = m.loc[1, 1.0], m.loc[1, 0.0], m.loc[0, 1.0]
print(f'AUC: {auc:.3f} | Recall: {vp / (vp + fn):.3f} | Precisão: {vp / (vp + fp):.3f}')
m.rename(index={0: 'Fica (real)', 1: 'Cancela (real)'}, columns={0.0: 'previu Fica', 1.0: 'previu Cancela'})

AUC: 0.856 | Recall: 0.842 | Precisão: 0.518


prediction,previu Fica,previu Cancela
label,,
Fica (real),711,277
Cancela (real),56,298


## 4. Comparação com o scikit-learn (notebook 02)

| | scikit-learn | Spark MLlib |
|---|---|---|
| Modelo | Regressão Logística, `class_weight='balanced'` | Regressão Logística, `weightCol` balanceado |
| Divisão | `train_test_split` estratificado 80/20 | `randomSplit` 80/20 |

Os números não são idênticos porque a divisão treino/teste é sorteada de forma diferente em cada biblioteca, mas ficam na mesma faixa: no scikit-learn, recall 0,797 e AUC 0,835; aqui, recall 0,842 e AUC 0,856. O ganho do Spark não é a métrica: é poder rodar o mesmo pipeline em bases que não cabem na memória de uma máquina.

In [7]:
spark.stop()